# mcphases Missingness Analysis
This notebook explores missing values for each CSV file in `mcphases/`, excluding files under `mcphases/merged/`.


## 1. Load libraries and set paths
Import libraries for data loading, analysis, and visualization, and define the repository path and target `mcphases` directory.


In [15]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno

# Navigate to the repository root (ML-for-Women)
current_notebook = Path('.').resolve()
if current_notebook.name == 'notebooks':
    repo_path = current_notebook.parent
else:
    # Navigate up until we find the mcphases directory
    repo_path = current_notebook
    while not (repo_path / 'mcphases').exists() and repo_path.parent != repo_path:
        repo_path = repo_path.parent

mcphases_dir = repo_path / 'mcphases'
output_dir = repo_path / 'notebooks' / 'missingness_outputs'
output_dir.mkdir(parents=True, exist_ok=True)

sns.set(style='whitegrid')

print(f'Repository path: {repo_path}')
print(f'mcphases directory: {mcphases_dir}')
print(f'mcphases exists: {mcphases_dir.exists()}')
print(f'Output directory: {output_dir}')


Repository path: D:\ML-for-Women
mcphases directory: D:\ML-for-Women\mcphases
mcphases exists: True
Output directory: D:\ML-for-Women\notebooks\missingness_outputs


## 2. Discover CSV files in mcphases excluding merged
Find every CSV file under `mcphases/`, then remove files located in `mcphases/merged/`.


In [12]:
csv_files = sorted(mcphases_dir.rglob('*.csv'))
csv_files = [path for path in csv_files if 'merged' not in path.parts]

len(csv_files), csv_files[:10]


(23,
 [WindowsPath('D:/ML-for-Women/mcphases/active_minutes.csv'),
  WindowsPath('D:/ML-for-Women/mcphases/active_zone_minutes.csv'),
  WindowsPath('D:/ML-for-Women/mcphases/altitude.csv'),
  WindowsPath('D:/ML-for-Women/mcphases/calories.csv'),
  WindowsPath('D:/ML-for-Women/mcphases/computed_temperature.csv'),
  WindowsPath('D:/ML-for-Women/mcphases/demographic_vo2_max.csv'),
  WindowsPath('D:/ML-for-Women/mcphases/distance.csv'),
  WindowsPath('D:/ML-for-Women/mcphases/estimated_oxygen_variation.csv'),
  WindowsPath('D:/ML-for-Women/mcphases/exercise.csv'),
  WindowsPath('D:/ML-for-Women/mcphases/glucose.csv')])

## 3. Inspect CSV schemas and row counts
Read the selected CSV files and summarize their column names, data types, and record counts.


In [13]:
overview = []

for path in csv_files:
    try:
        df = pd.read_csv(path)
    except Exception as exc:
        overview.append({
            'file': str(path.relative_to(repo_path)),
            'rows': None,
            'columns': None,
            'dtypes': None,
            'error': str(exc)
        })
        continue

    overview.append({
        'file': str(path.relative_to(repo_path)),
        'rows': len(df),
        'columns': len(df.columns),
        'dtypes': ', '.join(f'{col}:{dtype}' for col, dtype in df.dtypes.items()),
        'error': None,
    })

overview_df = pd.DataFrame(overview)
overview_df


,file,rows,columns,dtypes,error
0,mcphases\active_minutes.csv,5552,8,"id:int64, study_interval:int64, is_weekend:boo...",None
1,mcphases\active_zone_minutes.csv,154482,7,"id:int64, study_interval:int64, is_weekend:boo...",None
2,mcphases\altitude.csv,90878,6,"id:int64, study_interval:int64, is_weekend:boo...",None
3,mcphases\calories.csv,20166975,6,"id:int64, study_interval:int64, is_weekend:boo...",None
4,mcphases\computed_temperature.csv,5575,14,"id:int64, study_interval:int64, is_weekend:boo...",None
5,mcphases\demographic_vo2_max.csv,11482,8,"id:int64, study_interval:int64, is_weekend:boo...",None
6,mcphases\distance.csv,7666949,6,"id:int64, study_interval:int64, is_weekend:boo...",None
7,mcphases\estimated_oxygen_variation.csv,3070312,6,"id:int64, study_interval:int64, is_weekend:boo...",None
8,mcphases\exercise.csv,7282,26,"id:int64, study_interval:int64, is_weekend:boo...",None
9,mcphases\glucose.csv,837130,6,"id:int64, study_interval:int64, is_weekend:boo...",None


## 4. Compute missing value summary per file
Calculate the count and percentage of missing values for each column, then aggregate file-level summaries.


In [7]:
missingness_details = []
file_summaries = []

for path in csv_files:
    df = pd.read_csv(path)
    total_rows = len(df)
    missing_counts = df.isna().sum()
    missing_pct = (missing_counts / total_rows * 100).round(2) if total_rows else missing_counts

    for column in df.columns:
        missingness_details.append({
            'file': str(path.relative_to(repo_path)),
            'column': column,
            'missing_count': int(missing_counts[column]),
            'missing_pct': float(missing_pct[column]) if total_rows else None,
            'total_rows': total_rows,
        })

    file_summaries.append({
        'file': str(path.relative_to(repo_path)),
        'total_rows': total_rows,
        'total_columns': len(df.columns),
        'columns_with_missing': int((missing_counts > 0).sum()),
        'total_missing_cells': int(missing_counts.sum()),
        'pct_cells_missing': round(missing_counts.sum() / (total_rows * len(df.columns)) * 100, 2) if (total_rows > 0 and len(df.columns) > 0) else 0.0
    })

missingness_df = pd.DataFrame(missingness_details)
file_summary_df = pd.DataFrame(file_summaries)

# Ensure pct_cells_missing column exists and is numeric
if 'pct_cells_missing' not in file_summary_df.columns:
    file_summary_df['pct_cells_missing'] = 0.0
else:
    file_summary_df['pct_cells_missing'] = pd.to_numeric(file_summary_df['pct_cells_missing'], errors='coerce').fillna(0.0)

missingness_df.head(), file_summary_df


(Empty DataFrame
 Columns: []
 Index: [],
 Empty DataFrame
 Columns: [pct_cells_missing]
 Index: [])

## 5. Visualize missingness patterns
Create both file-level and column-level visualizations to highlight missing value distributions.


In [16]:
for i, path in enumerate(csv_files):
    df = pd.read_csv(path)
    file_name = path.name
    plt.figure(figsize=(14, 6))
    msno.matrix(df.sample(min(1000, len(df))))
    plt.title(f'Missing Data Matrix - {file_name}')
    plt.tight_layout()
    plt.savefig(output_dir / f'missingness_matrix_{i:02d}_{file_name}.png', dpi=100, bbox_inches='tight')
    plt.close()

print(f'Generated missingness matrices for {len(csv_files)} CSV files in {output_dir}')


C:\Users\caowe\AppData\Local\Temp\ipykernel_56444\2195312858.py:7: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
C:\Users\caowe\AppData\Local\Temp\ipykernel_56444\2195312858.py:7: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
C:\Users\caowe\AppData\Local\Temp\ipykernel_56444\2195312858.py:7: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
C:\Users\caowe\AppData\Local\Temp\ipykernel_56444\2195312858.py:7: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
C:\Users\caowe\AppData\Local\Temp\ipykernel_56444\2195312858.py:7: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layo

Generated missingness matrices for 23 CSV files in D:\ML-for-Women\notebooks\missingness_outputs


<Figure size 1400x600 with 0 Axes>

<Figure size 1400x600 with 0 Axes>

<Figure size 1400x600 with 0 Axes>

...

<Figure size 1400x600 with 0 Axes>

<Figure size 1400x600 with 0 Axes>

<Figure size 1400x600 with 0 Axes>

<Figure size 1400x600 with 0 Axes>

<Figure size 1400x600 with 0 Axes>